# BreadboardAOI - Qwen3-VL tutor runtime (Colab)

This notebook turns a free Colab GPU into the **cognitive loop** of the BreadboardAOI
workstation. Vision, motion gating and the KiCad DRC engine stay on your PC at 60 FPS;
only the Qwen3-VL diagnosis is offloaded here.

## How to run

1. **Runtime -> Change runtime type -> T4 GPU -> Save.**
2. **Runtime -> Run all.**
3. When cell 5 finishes it prints a single line that starts with `BBAOI1|`.
4. Open BreadboardAOI -> **REMOTE AI** -> paste that line into **PAIR CODE** -> **CONNECT**.

No Google sign-in and no Drive mount are needed in this default **manual** mode: the
notebook hands you one line, the app hands you a connected tutor. Set `USE_DRIVE = True`
in cell 1 only if you want the app to discover the tunnel automatically through Drive.

Weights come from this repository's `models-v1` GitHub Release, and `llama-server` is a
prebuilt CUDA binary from the official `ggml-org/llama.cpp` release, so a run is roughly
1 minute of setup plus the model download. Nothing is downloaded from your PC.

## What stays local

| Stage | Runs on |
| :--- | :--- |
| Camera capture, homography, hand gate, YOLO pose | your PC (60 FPS) |
| PinMatrix snapping, union-find netlist, KiCad DRC | your PC (sub-millisecond) |
| Voice synthesis and reflex alarms | your PC |
| Qwen3-VL diagnosis (one 256x256 crop + DRC JSON per fault) | this Colab runtime |


In [ ]:
# ---------------------------------------------------------------------------
# 1. Configuration - the defaults below are ready to run
# ---------------------------------------------------------------------------
MODEL_FILE = "Qwen3VL-8B-Instruct-Q4_K_M.gguf"
MMPROJ_FILE = "mmproj-Qwen3VL-8B-Instruct-F16.gguf"

# Weights are mirrored by .github/workflows/publish-models.yml into the
# models-v1 release of this repository, so Colab never contacts the publisher.
RELEASE_BASE = "https://github.com/wallali77666-cyber/BreadBoard-Assistant/releases/download/models-v1"
GITHUB_MODEL_URL = ""
MODEL_PARTS = [
    f"{RELEASE_BASE}/{MODEL_FILE}.part_00",
    f"{RELEASE_BASE}/{MODEL_FILE}.part_01",
    f"{RELEASE_BASE}/{MODEL_FILE}.part_02",
]
GITHUB_MMPROJ_URL = f"{RELEASE_BASE}/{MMPROJ_FILE}"

# Prebuilt llama-server from the official llama.cpp GitHub release. The build is
# pinned so the server flags used below stay valid; cell 3 falls back to a source
# build if the binary cannot start on this runtime.
LLAMA_BUILD = "b11922"
LLAMA_CUDA = "12.8"
LLAMA_ASSET = f"llama-{LLAMA_BUILD}-bin-ubuntu-cuda-{LLAMA_CUDA}-x64.tar.gz"
LLAMA_CUDART_ASSET = f"cudart-llama-{LLAMA_BUILD}-bin-ubuntu-cuda-{LLAMA_CUDA}-x64.tar.gz"
LLAMA_BASE = f"https://github.com/ggml-org/llama.cpp/releases/download/{LLAMA_BUILD}"

# 8089 is only used inside the runtime; the tunnel provides the public URL.
PORT = 8089

# False: the notebook prints a pair code you paste into the app (recommended).
# True: publish the endpoint to Google Drive so the app connects by itself.
USE_DRIVE = False

# Optional: force a specific bearer key instead of a random one. The app must use
# the same value, which the printed pair code always contains.
PRESET_API_KEY = ""

# 8B Q4_K_M fits a free T4 (16 GB VRAM) and runs fully on the GPU.
# The 30B-A3B needs Colab Pro (L4/A100) or the local RTX 3060 workstation.


In [ ]:
# ---------------------------------------------------------------------------
# 2. Runtime helpers: a resumable downloader, then the shared bearer key
# ---------------------------------------------------------------------------
import hashlib, json, os, secrets, shutil, time, urllib.error, urllib.request

# GitHub's CDN occasionally resets a long transfer. This downloader keeps the
# bytes already received, resumes with an HTTP Range request and retries with
# backoff, so a 1.9 GB part never restarts from zero.
def fetch_file(url, dest, attempts=8, label=""):
    dest = str(dest)
    parent = os.path.dirname(dest)
    if parent:
        os.makedirs(parent, exist_ok=True)

    for attempt in range(1, attempts + 1):
        existing = os.path.getsize(dest) if os.path.exists(dest) else 0
        request = urllib.request.Request(url, headers={"User-Agent": "BreadboardAOI-colab"})
        if existing:
            request.add_header("Range", f"bytes={existing}-")

        try:
            with urllib.request.urlopen(request, timeout=120) as response:
                resuming = existing > 0 and response.status == 206
                if not resuming:
                    existing = 0

                length = response.headers.get("Content-Length")
                total = existing + int(length) if length else None
                if attempt == 1:
                    print("GET", url, flush=True)

                mode = "ab" if resuming else "wb"
                with open(dest, mode) as handle:
                    while True:
                        chunk = response.read(1 << 20)
                        if not chunk:
                            break
                        handle.write(chunk)

            size = os.path.getsize(dest)
            if total is None or size >= total:
                print(f"    -> {label or dest} {size // (1024 * 1024)} MB", flush=True)
                return dest
            print(
                f"    incomplete {size // (1024 * 1024)}/{total // (1024 * 1024)} MB, resuming",
                flush=True,
            )
        except urllib.error.HTTPError as exc:
            if exc.code == 416:
                print(f"    -> {label or dest} already complete", flush=True)
                return dest
            print(f"    HTTP {exc.code} on attempt {attempt}/{attempts}", flush=True)
        except Exception as exc:
            print(f"    {type(exc).__name__}: {exc} (attempt {attempt}/{attempts})", flush=True)

        time.sleep(min(30, 2 * attempt))

    raise RuntimeError(f"could not download {url}; re-run this cell to resume")


def sha256_of(path):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for block in iter(lambda: handle.read(1 << 24), b""):
            digest.update(block)
    return digest.hexdigest()


def expected_hashes(sums_url):
    try:
        with urllib.request.urlopen(sums_url, timeout=60) as response:
            text = response.read().decode("utf-8", "replace")
    except Exception as exc:
        print("    checksum list unavailable:", exc, flush=True)
        return {}

    hashes = {}
    for line in text.splitlines():
        fields = line.split()
        if len(fields) == 2 and len(fields[0]) == 64:
            hashes[fields[1].lstrip("*")] = fields[0].lower()
    return hashes


session = {}
API_KEY = ""
SESSION_FILE = "/content/drive/MyDrive/breadboardaoi-colab.json"

if PRESET_API_KEY.strip():
    API_KEY = PRESET_API_KEY.strip()
    print("using the preset api key")

elif USE_DRIVE:
    # Optional path: adopt the key the app wrote to Drive and publish the tunnel
    # back into the same file once the server is serving.
    from google.colab import auth, drive

    auth.authenticate_user()
    drive.mount("/content/drive")

    if not os.path.exists(SESSION_FILE):
        raise SystemExit(
            "No session file in Drive. Press PUBLISH SESSION in the app first, "
            "or set USE_DRIVE = False to use a manual pair code."
        )

    with open(SESSION_FILE) as handle:
        session = json.load(handle)

    API_KEY = (session.get("apiKey") or "").strip() or secrets.token_urlsafe(32)
    print("drive mode: read", SESSION_FILE)

else:
    API_KEY = secrets.token_urlsafe(32)
    print("manual mode: generated a fresh api key")

print("api key:", API_KEY[:8] + "..." + API_KEY[-4:], f"({len(API_KEY)} chars)")
print("connect mode:", "google drive auto-connect" if USE_DRIVE else "manual pair code")


In [ ]:
# ---------------------------------------------------------------------------
# 3. Get llama-server - prebuilt CUDA build, source build only if it cannot run
# ---------------------------------------------------------------------------
import glob, os, subprocess, tarfile, urllib.request

CONTENT = "/content"
BIN_DIR = f"{CONTENT}/llama-bin"
SERVER_ENV = dict(os.environ)
SERVER = None


def fetch(url, dest):
    return fetch_file(url, dest, label=os.path.basename(dest))


def extract(archive, target):
    os.makedirs(target, exist_ok=True)
    with tarfile.open(archive) as tar:
        try:
            tar.extractall(target, filter="data")
        except TypeError:
            tar.extractall(target)


def find_server(root):
    hits = sorted(glob.glob(os.path.join(root, "**", "llama-server"), recursive=True))
    return hits[0] if hits else None


def server_works(path):
    try:
        proc = subprocess.run(
            [path, "--version"], capture_output=True, text=True, timeout=90, env=SERVER_ENV
        )
    except Exception as exc:
        print("probe failed:", exc, flush=True)
        return False

    text = (proc.stdout + proc.stderr).strip()
    if text:
        print("   ", text.splitlines()[0][:120], flush=True)
    return proc.returncode == 0


try:
    archive = fetch(f"{LLAMA_BASE}/{LLAMA_ASSET}", f"{CONTENT}/{LLAMA_ASSET}")
    extract(archive, BIN_DIR)
    SERVER = find_server(BIN_DIR)
except Exception as exc:
    print("prebuilt download failed:", exc, flush=True)

if SERVER and not server_works(SERVER):
    # Usually a missing CUDA runtime library; the official cudart bundle fixes it.
    try:
        cudart = fetch(f"{LLAMA_BASE}/{LLAMA_CUDART_ASSET}", f"{CONTENT}/{LLAMA_CUDART_ASSET}")
        extract(cudart, BIN_DIR)
        SERVER_ENV["LD_LIBRARY_PATH"] = BIN_DIR + ":" + SERVER_ENV.get("LD_LIBRARY_PATH", "")
        print("retrying with the bundled CUDA runtime", flush=True)
    except Exception as exc:
        print("cudart download failed:", exc, flush=True)

if not SERVER or not server_works(SERVER):
    print("--- building llama-server from source (5-10 min) ---", flush=True)
    subprocess.run(
        "apt-get -qq update && apt-get -qq install -y libcurl4-openssl-dev > /dev/null",
        shell=True, check=True,
    )
    subprocess.run(
        "rm -rf /content/llama.cpp && git clone --depth 1 "
        "https://github.com/ggml-org/llama.cpp /content/llama.cpp",
        shell=True, check=True,
    )
    subprocess.run(
        "cmake -S /content/llama.cpp -B /content/llama.cpp/build "
        "-DGGML_CUDA=ON -DCMAKE_BUILD_TYPE=Release -DLLAMA_CURL=OFF",
        shell=True, check=True,
    )
    subprocess.run(
        "cmake --build /content/llama.cpp/build --config Release -j 8 --target llama-server",
        shell=True, check=True,
    )
    SERVER = find_server("/content/llama.cpp/build")

if not SERVER:
    raise SystemExit("llama-server is unavailable; read the log above")

print("server:", SERVER, flush=True)


In [ ]:
# ---------------------------------------------------------------------------
# 4. Fetch the weights from this repository's models-v1 GitHub Release
# ---------------------------------------------------------------------------
import os, shutil

os.makedirs("/content/models", exist_ok=True)
model_path = f"/content/models/{MODEL_FILE}"
mmproj_path = f"/content/models/{MMPROJ_FILE}"
joining_path = model_path + ".joining"


def download(url, dest):
    return fetch_file(url, dest, label=os.path.basename(dest))


def join_parts(parts, dest):
    # Write to a scratch file and rename at the end, so an interrupted join can
    # never leave a truncated GGUF that looks complete to the next run.
    with open(joining_path, "wb") as output:
        for part in parts:
            with open(part, "rb") as source:
                shutil.copyfileobj(source, output, 1 << 24)
    os.replace(joining_path, dest)
    for part in parts:
        os.remove(part)


if not os.path.exists(model_path):
    if GITHUB_MODEL_URL:
        download(GITHUB_MODEL_URL, model_path)
    elif MODEL_PARTS:
        parts = [download(url, f"{model_path}.part{index:02d}") for index, url in enumerate(MODEL_PARTS)]
        join_parts(parts, model_path)
    else:
        raise SystemExit("No model URL configured: set GITHUB_MODEL_URL or MODEL_PARTS in cell 1.")

if not os.path.exists(mmproj_path):
    if GITHUB_MMPROJ_URL:
        download(GITHUB_MMPROJ_URL, mmproj_path)
    else:
        raise SystemExit("No mmproj configured: set GITHUB_MMPROJ_URL in cell 1.")

print("model:", model_path, os.path.getsize(model_path) // (1024 * 1024), "MB", flush=True)
print("mmproj:", mmproj_path, os.path.getsize(mmproj_path) // (1024 * 1024), "MB", flush=True)

# Integrity gate: the release publishes SHA256SUMS.txt next to the assets.
sums = expected_hashes(f"{RELEASE_BASE}/SHA256SUMS.txt")
for path, name in ((model_path, MODEL_FILE), (mmproj_path, MMPROJ_FILE)):
    expected = sums.get(name)
    if not expected:
        continue

    actual = sha256_of(path)
    if actual != expected:
        os.remove(path)
        raise SystemExit(
            f"{name} is corrupt (expected {expected[:12]}..., got {actual[:12]}...). "
            "The bad file was deleted; re-run this cell to download it again."
        )

    print("checksum ok:", name, flush=True)


In [ ]:
# ---------------------------------------------------------------------------
# 5. Serve llama-server through a cloudflared tunnel and print the pair code
# ---------------------------------------------------------------------------
import json, os, re, subprocess, time, urllib.request
from datetime import datetime, timezone

server_log_path = "/content/llama-server.log"
mmproj_args = ["--mmproj", mmproj_path] if os.path.exists(mmproj_path) else []

full_args = [
    "--model", model_path,
    *mmproj_args,
    "--host", "127.0.0.1", "--port", str(PORT),
    "--n-gpu-layers", "99",
    "--ctx-size", "4096",
    "--threads", "4", "--threads-batch", "8",
    "--flash-attn",
    "--cache-type-k", "q8_0", "--cache-type-v", "q8_0",
    "--api-key", API_KEY,
]

# Nightly binaries occasionally rename a tuning flag; this is the safe subset.
minimal_args = [
    "--model", model_path,
    *mmproj_args,
    "--host", "127.0.0.1", "--port", str(PORT),
    "--n-gpu-layers", "99",
    "--ctx-size", "4096",
    "--api-key", API_KEY,
]


def start_server(args):
    log = open(server_log_path, "w")
    proc = subprocess.Popen([SERVER, *args], stdout=log, stderr=subprocess.STDOUT, env=SERVER_ENV)
    print("llama-server pid", proc.pid, "args", len(args), flush=True)
    return proc


def healthy():
    # llama-server answers 503 while it loads and 200 once ready. With --api-key
    # the bearer token is checked on every route, so send it on the probe too.
    probe = urllib.request.Request(
        f"http://127.0.0.1:{PORT}/health",
        headers={"Authorization": f"Bearer {API_KEY}"},
    )
    try:
        with urllib.request.urlopen(probe, timeout=3) as response:
            return response.status == 200
    except Exception:
        return False


def wait_ready(proc, seconds):
    for _ in range(max(1, seconds // 2)):
        if healthy():
            return True
        if proc.poll() is not None:
            return False
        time.sleep(2)
    return False


server = start_server(full_args)
ready = wait_ready(server, 300)

if not ready:
    print("first launch failed; retrying with the minimal flag set", flush=True)
    with open(server_log_path) as handle:
        print(handle.read()[-3000:], flush=True)
    server = start_server(minimal_args)
    ready = wait_ready(server, 300)

if not ready:
    with open(server_log_path) as handle:
        print(handle.read()[-3000:], flush=True)
    raise SystemExit("llama-server did not become ready; read /content/llama-server.log")

print("llama-server is ready", flush=True)

if not os.path.exists("/usr/local/bin/cloudflared"):
    subprocess.run(
        "wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 "
        "-O /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared",
        shell=True,
        check=True,
    )

open("/content/cloudflared.log", "w").close()
tunnel = subprocess.Popen(
    [
        "/usr/local/bin/cloudflared", "tunnel",
        "--url", f"http://127.0.0.1:{PORT}",
        "--no-autoupdate",
        "--logfile", "/content/cloudflared.log",
    ],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)

tunnel_url = None
for _ in range(75):
    with open("/content/cloudflared.log") as handle:
        text = handle.read()
    match = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", text)
    if match:
        tunnel_url = match.group(0)
        break
    time.sleep(2)

if not tunnel_url:
    raise SystemExit("cloudflared did not publish a URL; read /content/cloudflared.log")

print("tunnel:", tunnel_url, flush=True)

# Confirm the public route reaches this server before handing the code over.
probe = urllib.request.Request(tunnel_url + "/health", headers={"Authorization": f"Bearer {API_KEY}"})
tunnel_ok = False
for attempt in range(20):
    try:
        with urllib.request.urlopen(probe, timeout=10) as response:
            if response.status == 200:
                tunnel_ok = True
                print("public tunnel is answering", flush=True)
                break
    except Exception as exc:
        print("waiting for the public tunnel:", exc, flush=True)
        time.sleep(3)

if not tunnel_ok:
    raise SystemExit("the tunnel is not reachable yet; re-run this cell once llama-server is ready")

PAIR_CODE = f"BBAOI1|{tunnel_url}|{API_KEY}"
with open("/content/pair-code.txt", "w") as handle:
    handle.write(PAIR_CODE + "\n")

banner = "=" * 78
print(banner)
print("PAIR CODE - copy the single line below into BreadboardAOI")
print("            REMOTE AI -> PAIR CODE -> CONNECT")
print()
print(PAIR_CODE)
print()
print("endpoint:", tunnel_url)
print(banner)

if USE_DRIVE:
    try:
        gpu = subprocess.check_output(
            ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], text=True
        ).strip()
    except Exception:
        gpu = "Colab runtime"

    session.update(
        {
            "status": "serving",
            "endpointUrl": tunnel_url,
            "apiKey": API_KEY,
            "model": MODEL_FILE,
            "gpu": gpu,
            "detail": f"llama-server :{PORT} via cloudflared quick tunnel",
            "updatedUtc": datetime.now(timezone.utc).isoformat(),
        }
    )

    with open(SESSION_FILE, "w") as handle:
        json.dump(session, handle, indent=2)

    print("published:", SESSION_FILE)


In [ ]:
# ---------------------------------------------------------------------------
# 6. Keep the session alive until you stop it
# ---------------------------------------------------------------------------
import time
from datetime import datetime, timezone

print("PAIR CODE (still valid):")
print(f"BBAOI1|{tunnel_url}|{API_KEY}")
print()

try:
    while True:
        time.sleep(60)
        server.poll()
        if server.returncode is not None:
            print("llama-server exited; the pair code is now dead", flush=True)
            if USE_DRIVE:
                session.update({"status": "stopped", "detail": "llama-server exited"})
                with open(SESSION_FILE, "w") as handle:
                    json.dump(session, handle, indent=2)
            break
        print("alive", datetime.now(timezone.utc).strftime("%H:%M:%S UTC"), flush=True)
except KeyboardInterrupt:
    print("stopped by operator")
    if USE_DRIVE:
        session.update(
            {
                "status": "stopped",
                "detail": "notebook stopped",
                "updatedUtc": datetime.now(timezone.utc).isoformat(),
            }
        )
        with open(SESSION_FILE, "w") as handle:
            json.dump(session, handle, indent=2)


# Troubleshooting

| Symptom | Fix |
| :--- | :--- |
| `ConnectionResetError` while downloading | Transient GitHub CDN reset. Re-run cell 4 (or cell 3): every download resumes with an HTTP Range request from the bytes already on disk, so nothing restarts from zero. |
| `checksum mismatch` | The join was corrupt; the bad file is deleted automatically. Re-run cell 4. |
| Cell 3 fails to download the binary | The pinned build tag may be retired; set `LLAMA_BUILD` to any tag from `github.com/ggml-org/llama.cpp/releases` that has `llama-*-bin-ubuntu-cuda-12.8-x64.tar.gz`. Cell 3 also falls back to a source build. |
| `llama-server did not become ready` | Read `/content/llama-server.log`. The most common cause is a CUDA driver mismatch; the source-build fallback solves it. |
| `cloudflared did not publish a URL` | Read `/content/cloudflared.log`. Colab occasionally blocks quick tunnels; re-run cell 5. |
| App says `not answering yet` | The tunnel is up but the model is still loading. Wait for `llama-server is ready`, then press CONNECT again. |
| `401` from the tunnel | The key in the pair code does not match the running server. Re-copy the line printed by cell 5. |
| Runtime disconnected | Free Colab stops after idle/12 h. `Run all` again and paste the new pair code; the old tunnel is dead. |

## Free T4 limits

16 GB VRAM, ~12.7 GB RAM. Qwen3-VL-8B Q4_K_M (4.8 GB) plus the F16 projector (1.1 GB)
fits comfortably. Qwen3-VL-30B-A3B is 17.7 GB of weights and cannot run here - keep it on
the RTX 3060 workstation.
